# C01 — 캐리어 28번 에어 누수 (2026-10-17, X-029)

정리본: [`cases/C01.md`](cases/C01.md). 이 노트북은 그 근거를 다시 만든다 (DB 는 캐시가 있으면 안 감).

| 단계 | 내용 |
|---|---|
| 1 | 일지 내용 |
| 2 | 가동일 기록(하루 단위)을 운영 판정 함수에 넣기 — 그날 모델이 내렸을 판정 |
| 3 | DB → 운영 코어 재생 (10-16 ~ 10-27) — 통과 하나하나 |
| 4 | 모델 화면에 떴을 내용 (재생의 판정 결과) |
| 5 | 실제 엣지 경고 (10-26 W0073) |
| 6 | 유닛 사이클을 대차별로 — 유닛 지표가 이 대차를 보나 |

In [ ]:
import sys, json; sys.path.insert(0, "..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from review_lib import paths, db, replay, backend, plot
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
FIGS = paths.CHECK_DIRS["conveyor"] / "figs"; FIGS.mkdir(exist_ok=True)
WAGON, SENSORS = 28, ["P00451", "P00459"]          # 전진·후진 Unit 캐리어 진입감지
ACT = (pd.Timestamp("2026-10-17 09:30"), pd.Timestamp("2026-10-17 10:30"))

## 1. 일지

In [ ]:
acts = pd.read_csv(paths.ACTIONS_DIR / "actions_conveyor.csv")
acts[acts.incident_id == "C01"].T

## 2. 가동일 기록 → 운영 판정 (하루 단위)

`v2_2_1/backend/data/conveyor_core_history.json` — 09-10 ~ 10-19 씨앗(배치), 10-26 ~ 11-05 실시간 누적. **10-20 ~ 10-25 는 비어 있다.**

In [ ]:
backend.use("v2.2")
from tools_yeji.conveyor_models.core_judge import judge, sensor_level
from tools_yeji.conveyor_models.core_params import LONG_REF
H = pd.DataFrame(json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json")))
H["day"] = pd.to_datetime(H.day)
J = judge(H)
w = J[(J.wagon == WAGON) & J.sensor.isin(SENSORS) & (J.day >= "2026-09-08")]
w.pivot_table(index="day", columns="sensor", values=["state", "rearm", "width_ex", "rise_off"], aggfunc="first").round(2)

In [ ]:
# 같은 날 후진 진입(P00459)에서 이상인 대차 — 10-17·10-19 은 28번 하나뿐
x = J[(J.sensor == "P00459") & (J.day.between("2026-10-10", "2026-10-31")) & (J.state != "정상")]
x.groupby("day").apply(lambda g: ", ".join(f"{a}:{b}" for a, b in zip(g.wagon, g.state)), include_groups=False)

## 3. DB → 운영 코어 재생 (통과 하나하나)

기록은 10-16 이전 날만 넘긴다 → 10-16 부터 모델이 돌았다면 무엇을 봤을지.
재생 결과는 가동일 기록과 맞는다 (10-16·17 검증: 앞 끝 위치 동일, 재감지 차이 ≤ 0.011).

In [ ]:
TAGS = db.model_tags("conveyor", "v2.2")
raw = db.fetch_range("2026-10-16", "2026-10-27", TAGS, group="conveyor")
hist = json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json"))
P, E, core = replay.conveyor_core(raw, history=[r for r in hist if r["day"] < "2026-10-16"],
                                  eval_every="15min", units=("md_fwd", "md_rev"))
P["day"] = (P.t - pd.Timedelta(hours=6)).dt.normalize()
D = P.groupby(["day", "sensor"]).dist.median().rename("D")        # 그날 센서의 대차 공통 위치
P = P.join(D, on=["day", "sensor"]); P["front"] = P.dist - P.D      # 앞 끝 편차 (−: 표식보다 앞에서 켜짐)
p28 = P[(P.wagon == WAGON) & P.sensor.isin(SENSORS)].sort_values("t")
p28.groupby([p28.t.dt.date, "sensor"]).agg(n=("t", "size"), rearm=("rearm", "mean"),
                                          front=("front", "median"), width=("width", "median")).unstack().round(2)

In [ ]:
# 바뀐 순간: 10-16 마지막 통과 vs 10-17 첫 통과, 10-22 마지막 재감지 vs 이후
r = p28[p28.sensor == "P00459"]
print("10-16 마지막 통과", r[r.t < "2026-10-17 06:00"].tail(2)[["t", "rearm", "front"]].to_string())
print("10-17 첫 통과   ", r[r.t > "2026-10-17 05:00"].head(3)[["t", "rearm", "front"]].to_string())
last_dirty = r[(r.t < "2026-10-23") & (r.rearm > 0)].t.max()
print("마지막 재감지 통과", last_dirty, "→ 그 뒤 첫 통과", r[r.t > last_dirty].t.min())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
for ax, s in zip(axes, SENSORS):
    q = p28[p28.sensor == s]
    others = P[(P.sensor == s) & (P.wagon != WAGON)].set_index("t").rearm.resample("1h").mean()
    ax.plot(others.index, others.values, color="gray", lw=0.8, label="다른 대차 평균 (1시간)")
    ax.scatter(q.t, q.rearm, s=8, color="tab:red", label=f"대차 {WAGON} 통과별 재감지 수")
    ax.axvspan(*ACT, color="tab:blue", alpha=0.3, label="일지 조치 10-17 09:30~10:30")
    ax.axvspan(pd.Timestamp("2026-10-22 09:56"), pd.Timestamp("2026-10-22 11:34"), color="tab:green", alpha=0.3,
               label="재감지 사라짐 10-22 (일지 없음)")
    ax.axvline(pd.Timestamp("2026-10-26 07:09"), color="k", ls=":", label="엣지 경고 W0073 10-26 07:09")
    ax.set_ylabel(s); ax.set_ylim(-0.2, 4)
axes[0].legend(loc="upper right", fontsize=8, ncol=3)
axes[0].set_title(f"대차 {WAGON} — 전진(P00451)·후진(P00459) Unit 캐리어 진입감지 재감지")
fig.tight_layout(); fig.savefig(FIGS / "C01_wagon28_rearm.png", dpi=110, bbox_inches="tight")

In [ ]:
# 10-16 ~ 10-17 확대: 앞 끝 편차 — 매 통과 같은 자리(후진 −6, 전진 −14)에서 먼저 켜짐 = 대차에 붙은 고정된 무언가
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
for ax, s in zip(axes, SENSORS):
    q = p28[(p28.sensor == s) & p28.t.between("2026-10-16 06:00", "2026-10-18 01:00")]
    ax.scatter(q.t, q.front, s=10, c=np.where(q.rearm > 0, "tab:red", "tab:gray"))
    ax.axvspan(*ACT, color="tab:blue", alpha=0.3)
    ax.set_ylabel(f"{s}\n앞 끝 편차"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_title("대차 28 앞 끝 편차 (빨강 = 그 통과에서 재감지)")
fig.tight_layout(); fig.savefig(FIGS / "C01_wagon28_front.png", dpi=110, bbox_inches="tight")

## 4. 모델 화면에 떴을 내용 (재생 판정, 15분마다)

- P00451(전진)은 `센서 재감지`(대차 20대 넘게 재감지) 플래그가 늘 켜져 있어 **대차 경고가 전부 센서 경고 하나로 묶인다** → 28번 변화는 따로 안 보인다.
- P00459(후진)는 28번만 `이물(분리)` → 점검 추천(등급 2) 경고 `wagon_rearm`.

In [ ]:
w28 = E[E.wagon == WAGON]
print(w28.groupby(["sensor", "state"]).t.agg(["min", "max", "count"]))
fl = E[E.wagon.isna() & (E["flags"].fillna("") != "")]
print(fl.groupby(["sensor", "flags"]).t.agg(["min", "max", "count"]))

## 5. 실제 엣지 경고 — W0073 (10-26 07:09 ~ 11:43)

`since: "10-17"` — 엣지 가동일 기록이 10-19 씨앗에서 멈춰 있어(10-20~25 기록 없음) 10-17·10-19 의 이물 상태를 이어받았다.
같은 조건(씨앗 기록 + 10-26 02:15 시작)으로 재생해도 이 경고는 **재현되지 않는다** — 그날 28번 통과는 전부 깨끗했다. 엣지 내부 상태 확인이 필요하다.

In [ ]:
ev = pd.read_csv(paths.EPISODES_DIR / "warning_events.csv", low_memory=False, parse_dates=["ts"])
ev[(ev.key == "P00459:wagons") & ev.ts.between("2026-10-26", "2026-10-27")][["ts", "event", "active", "message", "since"]].head(12)

## 6. 유닛 사이클을 대차별로 — 유닛 지표가 28번을 보나

운영 유닛 모델은 하루 평균만 쓴다. 사이클마다 직전 진입감지 통과의 대차를 붙여(지연 2.5–3.6초로 한 덩어리) 28번과 나머지를 나눈다.

In [ ]:
C = core.unit_cycles
cols = ["stop_fwd_time", "stop_rev_time", "catch_cnt", "drag_cnt", "air_time", "return_time", "nozzle_rev_time"]
out = []
for u in ["md_fwd", "md_rev"]:
    a = replay.attach_wagon(C, P, u)
    a["day"] = (a.t - pd.Timedelta(hours=6)).dt.date
    a["grp"] = np.where(a.wagon == WAGON, "28", "나머지")
    m = a[a.day.astype(str).isin(["2026-10-16", "2026-10-17", "2026-10-21", "2026-10-23"])].groupby(["day", "grp"])[cols].mean()
    out.append(m.assign(unit=u))
pd.concat(out).round(3)